## 1. Batch Normalisation

Batch Normalisation is a technique used to improve the training of deep neural networks by normalising the inputs of each layer. It works by standardising the activations of the previous layer for each mini-batch, which helps to stabilise and accelerate the learning process. By reducing internal covariate shift, batch normalisation allows for higher learning rates, reduces sensitivity to initialisation, and can act as a form of regularisation, sometimes reducing the need for dropout.

# Imports

In [22]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:0


In [23]:
# Moderated augmentation to target ~92–97% accuracy
transform_train = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

trainset = torchvision.datasets.MNIST(root='./data', train=True,
                                      download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32,
                                         shuffle=True, num_workers=2)

testset = torchvision.datasets.MNIST(root='./data', train=False,
                                      download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=32,
                                         shuffle=False, num_workers=2)

classes = ('0', '1', '2', '3', '4', '5', '6', '7', '8', '9')

# model

In [24]:
# Model 1: with BatchNorm and moderate Dropout
class ModelBN(nn.Module):
    def __init__(self):
        super(ModelBN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(128 * 7 * 7, 128)
        self.bn4 = nn.BatchNorm1d(128)
        self.dropout2 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.dropout(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.bn4(self.fc1(x)))
        x = self.dropout2(x)
        x = self.fc2(x)
        return x


# Model 2: same config but WITHOUT BatchNorm (moderate Dropout)
class ModelNoBN(nn.Module):
    def __init__(self):
        super(ModelNoBN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(128 * 7 * 7, 128)
        self.dropout2 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = F.relu(self.conv3(x))
        x = self.dropout(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        return x

### 4. Train the Network

In [25]:
# Train both models: with and without BatchNorm

def train_model(net, trainloader, testloader, device, epochs=8, l1_lambda=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    for epoch in range(epochs):
        running_loss = 0.0
        net.train()
        for i, data in enumerate(trainloader, 0):
            inputs, labels = data[0].to(device), data[1].to(device)

            optimizer.zero_grad()

            try:
                outputs = net(inputs)
                loss = criterion(outputs, labels)

                # L1 regularization
                l1_norm = sum(p.abs().sum() for p in net.parameters())
                loss = loss + l1_lambda * l1_norm

                loss.backward()
                optimizer.step()

                running_loss += loss.item()
                if i % 100 == 99:
                    print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 100:.3f}')
                    running_loss = 0.0
            except RuntimeError as e:
                print(f"A runtime error occurred: {e}")
                print("This is likely due to an incorrect linear layer size.")
                print("To fix the linear layer size, print(x.shape) in the model's forward pass and update the fc1 layer accordingly.")
                return net

        # simple validation loss
        val_loss = 0.0
        net.eval()
        with torch.no_grad():
            for data in testloader:
                images, labels = data[0].to(device), data[1].to(device)
                outputs = net(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        val_loss /= len(testloader)
        print(f'Epoch {epoch + 1}, Validation Loss: {val_loss:.4f}')

        scheduler.step()

    return net

print("Training Model with Batch Normalisation...")
net_bn = ModelBN().to(device)
net_bn = train_model(net_bn, trainloader, testloader, device)

print("\nTraining Model WITHOUT Batch Normalisation...")
net_no_bn = ModelNoBN().to(device)
net_no_bn = train_model(net_no_bn, trainloader, testloader, device)

print("Finished Training both models")

Training Model with Batch Normalisation...
[1,   100] loss: 5.380
[1,   200] loss: 3.211
[1,   300] loss: 3.035
[1,   400] loss: 2.869
[1,   500] loss: 2.970
[1,   600] loss: 2.826
[1,   700] loss: 2.748
[1,   800] loss: 2.652
[1,   900] loss: 2.629
[1,  1000] loss: 2.539
[1,  1100] loss: 2.522
[1,  1200] loss: 2.314
[1,  1300] loss: 2.363
[1,  1400] loss: 2.249
[1,  1500] loss: 2.089
[1,  1600] loss: 2.109
[1,  1700] loss: 2.038
[1,  1800] loss: 1.856
Epoch 1, Validation Loss: 0.0829
[2,   100] loss: 1.908
[2,   200] loss: 1.803
[2,   300] loss: 1.769
[2,   400] loss: 1.738
[2,   500] loss: 1.814
[2,   600] loss: 1.631
[2,   700] loss: 1.683
[2,   800] loss: 1.557
[2,   900] loss: 1.570
[2,  1000] loss: 1.640
[2,  1100] loss: 1.669
[2,  1200] loss: 1.575
[2,  1300] loss: 1.575
[2,  1400] loss: 1.573
[2,  1500] loss: 1.498
[2,  1600] loss: 1.519
[2,  1700] loss: 1.489
[2,  1800] loss: 1.458
Epoch 2, Validation Loss: 0.0787
[3,   100] loss: 1.416
[3,   200] loss: 1.440
[3,   300] loss: 

In [26]:
def evaluate_model(net, dataloader, device, name="model"):
    correct = 0
    total = 0
    net.eval()
    with torch.no_grad():
        for data in dataloader:
            images, labels = data[0].to(device), data[1].to(device)
            outputs = net(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    if total > 0:
        acc = 100.0 * correct / total
        print(f'Accuracy of {name} on the 10000 test images: {acc:.2f} %')
    else:
        print(f'No samples to evaluate for {name}.')

# Evaluate both models
evaluate_model(net_bn, testloader, device, name="Model with BatchNorm")
evaluate_model(net_no_bn, testloader, device, name="Model without BatchNorm")

Accuracy of Model with BatchNorm on the 10000 test images: 98.40 %
Accuracy of Model without BatchNorm on the 10000 test images: 97.32 %
